# IPCA — acumulado em 12 meses

Este notebook é o motor de coleta e publicação da série `ipca-acum-12m`.

- **Fonte:** IBGE, via API de Dados Abertos do Banco Central (SGS), série **13522** — “IPCA, variação acumulada em 12 meses (%)”.
- **Endpoint:** `https://api.bcb.gov.br/dados/serie/bcdata.sgs.13522/dados?formato=json&dataInicial=01/01/1995&dataFinal=<hoje>`
- **Definição:** variação percentual do IPCA acumulada nos últimos 12 meses (taxa efetiva do período), unidade `percentage` (`%`, 2 casas decimais), granularidade `monthly`.
- **Fórmula:** nenhuma. O cálculo é definido pela fonte; o notebook apenas busca a série oficial, filtra os meses fechados e formata.
- **Período publicado:** começa em `1995-01-01` e termina no último mês fechado disponível.
- **Arquivo gerado:** `ipca/ipca-acum-12m.json`, com `schema: 1` e `id: ipca-acum-12m`.

## Decisão metodológica

A revisão registrada em `docs/metodologia.md` aprovou publicar diretamente a série oficial **SGS 13522**, sem recalcular a composição a partir da SGS 433 (variação mensal). A 433 não participa da saída. Como a coleta começa em `1995-01-01`, a série publicada ganha os **11 pontos de 1995-01 a 1995-11** antes do início anterior em `1995-12`, e passa a se estender até o último mês fechado.


## Coleta da série oficial

A leitura é feita desde `1995-01-01`, sem atualização incremental e sem ler o JSON já publicado para decidir o período.


In [ ]:
from datetime import date, datetime
from pathlib import Path
import json
import os

import pandas as pd
import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
PASTA = RAIZ / "ipca" if (RAIZ / "ipca").is_dir() else RAIZ
ARQUIVO = PASTA / "ipca-acum-12m.json"

SERIE = 13522
URL = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{SERIE}/dados"
INICIO = date(1995, 1, 1)
HOJE = date.today()

resposta = requests.get(
    URL,
    params={
        "formato": "json",
        "dataInicial": INICIO.strftime("%d/%m/%Y"),
        "dataFinal": HOJE.strftime("%d/%m/%Y"),
    },
    timeout=60,
)
resposta.raise_for_status()
bruto = resposta.json()

if not bruto:
    raise ValueError(f"a série SGS {SERIE} não retornou dados")

df = pd.DataFrame(
    {
        "data": pd.to_datetime(
            [datetime.strptime(item["data"], "%d/%m/%Y") for item in bruto]
        ),
        "valor": [float(item["valor"]) for item in bruto],
    }
).sort_values("data", ignore_index=True)

print(f"Série SGS {SERIE}: {len(df)} leituras")
print(f"Período bruto: {df['data'].min():%Y-%m-%d} a {df['data'].max():%Y-%m-%d}")
print(f"Arquivo de saída: {ARQUIVO}")
print("\nPrimeiras 5 leituras:")
print(df.head().to_string(index=False))
print("\nÚltimas 5 leituras:")
print(df.tail().to_string(index=False))


## Filtragem dos meses fechados

O mês corrente é descartado porque ainda está aberto. A fonte devolve as datas no primeiro dia de cada mês; essa convenção é preservada.


In [ ]:
primeiro_dia_mes_corrente = pd.Timestamp(HOJE.replace(day=1))
fechados = df[df["data"] < primeiro_dia_mes_corrente].copy()

if fechados.empty:
    raise ValueError("a fonte não retornou nenhum mês fechado")

if fechados["data"].dt.to_period("M").duplicated().any():
    raise ValueError("a fonte retornou meses repetidos")

print(f"Mês corrente excluído: {HOJE:%Y-%m}")
print(f"Meses fechados publicados: {len(fechados)}")
print(
    f"Primeiro: {fechados['data'].iloc[0]:%Y-%m-%d} "
    f"({fechados['valor'].iloc[0]:.2f}%)"
)
print(
    f"Último:   {fechados['data'].iloc[-1]:%Y-%m-%d} "
    f"({fechados['valor'].iloc[-1]:.2f}%)"
)


## Escrita do JSON canônico

Os pontos são gravados como `{"date": "YYYY-MM-DD", "value": ...}`, em ordem ascendente e sem campos duplicados.


In [ ]:
pontos = [
    {"date": data.strftime("%Y-%m-%d"), "value": float(valor)}
    for data, valor in zip(fechados["data"], fechados["valor"])
]

serie = {
    "schema": 1,
    "id": "ipca-acum-12m",
    "unit": {"type": "percentage", "symbol": "%", "decimals": 2},
    "granularity": "monthly",
    "source": "IBGE",
    "data": pontos,
}

with ARQUIVO.open("w", encoding="utf-8") as arquivo:
    json.dump(serie, arquivo, ensure_ascii=False, indent=2)

print(f"{len(pontos)} pontos escritos em {ARQUIVO.name}")
print(f"Primeiro ponto: {pontos[0]}")
print(f"Último ponto:   {pontos[-1]}")


## Validação inline

A célula final falha se o contrato, a unicidade dos meses, a ordem, a finitude dos valores ou a igualdade com a fonte forem violados. Também confere os números citados em `docs/metodologia.md`.


In [ ]:
import math

esperado = {
    "schema": 1,
    "id": "ipca-acum-12m",
    "unit": {"type": "percentage", "symbol": "%", "decimals": 2},
    "granularity": "monthly",
    "source": "IBGE",
}
for campo, valor_esperado in esperado.items():
    assert serie[campo] == valor_esperado, (campo, serie[campo], valor_esperado)

datas = [ponto["date"] for ponto in serie["data"]]
meses = [data[:7] for data in datas]

assert datas, "série vazia"
assert datas == sorted(datas), "datas fora de ordem"
assert len(meses) == len(set(meses)), "mês repetido"
assert datas[0] == "1995-01-01", datas[0]
assert datas[-1][:7] < f"{HOJE:%Y-%m}", datas[-1]
assert len(datas) == len(fechados), (len(datas), len(fechados))

for ponto in serie["data"]:
    assert set(ponto) == {"date", "value"}, sorted(ponto)
    assert not isinstance(ponto["value"], bool), ponto
    assert isinstance(ponto["value"], (int, float)), ponto
    assert math.isfinite(ponto["value"]), ponto

esperado_da_fonte = [
    {"date": data.strftime("%Y-%m-%d"), "value": float(valor)}
    for data, valor in zip(fechados["data"], fechados["valor"])
]
assert serie["data"] == esperado_da_fonte, "valor publicado difere da fonte"

por_data = {ponto["date"]: ponto["value"] for ponto in serie["data"]}
assert por_data["2025-06-01"] == 5.35, por_data["2025-06-01"]
assert por_data["2002-12-01"] == 12.53, por_data["2002-12-01"]
assert por_data["2003-02-01"] == 15.85, por_data["2003-02-01"]
assert datas[:12] == [f"1995-{mes:02d}-01" for mes in range(1, 13)], datas[:12]

with ARQUIVO.open(encoding="utf-8") as arquivo:
    salvo = json.load(arquivo)
assert salvo == serie, "arquivo gravado difere do objeto validado"

print(
    f"ok: {len(datas)} meses únicos, {datas[0]} a {datas[-1]}, "
    f"valores idênticos à SGS 13522"
)
